In [37]:
import pandas as pd
import re
from collections import defaultdict

In [2]:
df = pd.read_csv("time_series_60min_singleindex_filtered.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50401 entries, 0 to 50400
Columns: 300 entries, utc_timestamp to UA_load_forecast_entsoe_transparency
dtypes: float64(298), object(2)
memory usage: 115.4+ MB


In [3]:
df.head()

,utc_timestamp,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual,BE_load_actual_entsoe_transparency,BE_load_forecast_entsoe_transparency,BE_solar_generation_actual,...,SI_load_actual_entsoe_transparency,SI_load_forecast_entsoe_transparency,SI_solar_generation_actual,SI_wind_onshore_generation_actual,SK_load_actual_entsoe_transparency,SK_load_forecast_entsoe_transparency,SK_solar_generation_actual,SK_wind_onshore_generation_actual,UA_load_actual_entsoe_transparency,UA_load_forecast_entsoe_transparency
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,5946.0,6701.0,35.0,NaN,69.0,9484.0,9897.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,5726.0,6593.0,45.0,NaN,64.0,9152.0,9521.0,NaN,...,1045.47,816.0,NaN,1.17,2728.0,2860.0,3.8,NaN,NaN,NaN
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,5347.0,6482.0,41.0,NaN,65.0,8799.0,9135.0,NaN,...,1004.79,805.0,NaN,1.04,2626.0,2810.0,3.8,NaN,NaN,NaN
4,2015-01-01T03:00:00Z,2015-01-01T04:00:00+0100,5249.0,6454.0,38.0,NaN,64.0,8567.0,8909.0,NaN,...,983.79,803.0,NaN,1.61,2618.0,2780.0,3.8,NaN,NaN,NaN


In [7]:
dummy = df.columns.tolist()
selected_columns = []

selected_columns.append(dummy[:2])
selected_columns

[['utc_timestamp', 'cet_cest_timestamp']]

In [10]:
dummy[2:7]

['AT_load_actual_entsoe_transparency',
 'AT_load_forecast_entsoe_transparency',
 'AT_price_day_ahead',
 'AT_solar_generation_actual',
 'AT_wind_onshore_generation_actual']

In [13]:
dummy[8:13]

['BE_load_forecast_entsoe_transparency',
 'BE_solar_generation_actual',
 'BE_wind_generation_actual',
 'BE_wind_offshore_generation_actual',
 'BE_wind_onshore_generation_actual']

In [14]:
dummy[14:19]

['BG_load_forecast_entsoe_transparency',
 'BG_solar_generation_actual',
 'BG_wind_onshore_generation_actual',
 'CH_load_actual_entsoe_transparency',
 'CH_load_forecast_entsoe_transparency']

In [15]:
areas = dummy[2:]
areas

['AT_load_actual_entsoe_transparency',
 'AT_load_forecast_entsoe_transparency',
 'AT_price_day_ahead',
 'AT_solar_generation_actual',
 'AT_wind_onshore_generation_actual',
 'BE_load_actual_entsoe_transparency',
 'BE_load_forecast_entsoe_transparency',
 'BE_solar_generation_actual',
 'BE_wind_generation_actual',
 'BE_wind_offshore_generation_actual',
 'BE_wind_onshore_generation_actual',
 'BG_load_actual_entsoe_transparency',
 'BG_load_forecast_entsoe_transparency',
 'BG_solar_generation_actual',
 'BG_wind_onshore_generation_actual',
 'CH_load_actual_entsoe_transparency',
 'CH_load_forecast_entsoe_transparency',
 'CH_solar_capacity',
 'CH_solar_generation_actual',
 'CH_wind_onshore_capacity',
 'CH_wind_onshore_generation_actual',
 'CY_load_actual_entsoe_transparency',
 'CY_load_forecast_entsoe_transparency',
 'CY_wind_onshore_generation_actual',
 'CZ_load_actual_entsoe_transparency',
 'CZ_load_forecast_entsoe_transparency',
 'CZ_solar_generation_actual',
 'CZ_wind_onshore_generation_act

In [16]:
c = areas[0]
c[:2]

'AT'

In [30]:
code_area = []

for col in areas:
    if col[:2] == "DE":
        # will handle this later bcs there are so many types of DE
        continue
    if col[:2] not in code_area:
        code_area.append(col[:2])
    else:
        continue

print(f"Total number of code area: {len(code_area)}")
print(f"List of code area: {code_area}")

Total number of code area: 31
List of code area: ['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UA']


In [27]:
DE_area = []

for col in areas:
    if col[:2] == "DE":
        DE_area.append(col)  

In [26]:
pattern

'load|solar|wind|price|capacity|generation'

In [31]:
keywords = ['load', 'solar', 'wind', 'price', 'capacity', 'generation']
pattern = '|'.join(keywords)

for col in DE_area:
    # Cari posisi keyword pertama kali muncul
    match = re.search(f"_({pattern})", col)
    if match:
        # Ambil string sebelum underscore keyword tersebut
        area_code = col[:match.start()]
        if area_code not in code_area:
            code_area.append(area_code)
        else:
            continue
    else:
        continue

print(f"Total number of code area: {len(code_area)}")
print(f"List of code area: {code_area}")

Total number of code area: 37
List of code area: ['AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'ME', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'UA', 'DE', 'DE_50hertz', 'DE_LU', 'DE_amprion', 'DE_tennet', 'DE_transnetbw']


In [35]:
x = [i for i in areas if i[:2] == 'AT']

In [36]:
x

['AT_load_actual_entsoe_transparency',
 'AT_load_forecast_entsoe_transparency',
 'AT_price_day_ahead',
 'AT_solar_generation_actual',
 'AT_wind_onshore_generation_actual']

In [43]:
df_AT__ = df[['utc_timestamp', 'cet_cest_timestamp']]
df_AT__ = pd.concat([df_AT__, df[[i for i in areas if i[:2] == 'AT']]], axis=1)
df_AT__.head()

,utc_timestamp,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,5946.0,6701.0,35.0,NaN,69.0
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,5726.0,6593.0,45.0,NaN,64.0
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,5347.0,6482.0,41.0,NaN,65.0
4,2015-01-01T03:00:00Z,2015-01-01T04:00:00+0100,5249.0,6454.0,38.0,NaN,64.0


In [45]:
df_each_area = defaultdict(pd.DataFrame)

In [46]:
df_each_area

defaultdict(pandas.core.frame.DataFrame, {})

In [52]:
df_each_area = {}

for code in code_area:
    dummy_df = df[['utc_timestamp', 'cet_cest_timestamp']]
    dummy_val = df[[i for i in areas if i[:2] == code]]
    dummy_df = pd.concat([dummy_df, dummy_val], axis=1)

    key_code = 'df_' + code
    dummy_dict = {
        key_code: dummy_df
    }

    df_each_area |= dummy_dict

In [56]:
df_each_area.keys()

dict_keys(['df_AT', 'df_BE', 'df_BG', 'df_CH', 'df_CY', 'df_CZ', 'df_DK', 'df_EE', 'df_ES', 'df_FI', 'df_FR', 'df_GB', 'df_GR', 'df_HR', 'df_HU', 'df_IE', 'df_IT', 'df_LT', 'df_LU', 'df_LV', 'df_ME', 'df_NL', 'df_NO', 'df_PL', 'df_PT', 'df_RO', 'df_RS', 'df_SE', 'df_SI', 'df_SK', 'df_UA', 'df_DE', 'df_DE_50hertz', 'df_DE_LU', 'df_DE_amprion', 'df_DE_tennet', 'df_DE_transnetbw'])

In [62]:
for i, j in df_each_area.items():
    print(j)

              utc_timestamp        cet_cest_timestamp  \
0      2014-12-31T23:00:00Z  2015-01-01T00:00:00+0100   
1      2015-01-01T00:00:00Z  2015-01-01T01:00:00+0100   
2      2015-01-01T01:00:00Z  2015-01-01T02:00:00+0100   
3      2015-01-01T02:00:00Z  2015-01-01T03:00:00+0100   
4      2015-01-01T03:00:00Z  2015-01-01T04:00:00+0100   
...                     ...                       ...   
50396  2020-09-30T19:00:00Z  2020-09-30T21:00:00+0200   
50397  2020-09-30T20:00:00Z  2020-09-30T22:00:00+0200   
50398  2020-09-30T21:00:00Z  2020-09-30T23:00:00+0200   
50399  2020-09-30T22:00:00Z  2020-10-01T00:00:00+0200   
50400  2020-09-30T23:00:00Z  2020-10-01T01:00:00+0200   

       AT_load_actual_entsoe_transparency  \
0                                     NaN   
1                                  5946.0   
2                                  5726.0   
3                                  5347.0   
4                                  5249.0   
...                                   ...   


In [63]:
save_dataset_path = "dataset/"

for code, data in df_each_area.items():
    filename = save_dataset_path + code + '.csv'
    data.to_csv(filename, index=False)

In [64]:
df_AT__

,utc_timestamp,cet_cest_timestamp,AT_load_actual_entsoe_transparency,AT_load_forecast_entsoe_transparency,AT_price_day_ahead,AT_solar_generation_actual,AT_wind_onshore_generation_actual
0,2014-12-31T23:00:00Z,2015-01-01T00:00:00+0100,NaN,NaN,NaN,NaN,NaN
1,2015-01-01T00:00:00Z,2015-01-01T01:00:00+0100,5946.0,6701.0,35.0,NaN,69.0
2,2015-01-01T01:00:00Z,2015-01-01T02:00:00+0100,5726.0,6593.0,45.0,NaN,64.0
3,2015-01-01T02:00:00Z,2015-01-01T03:00:00+0100,5347.0,6482.0,41.0,NaN,65.0
4,2015-01-01T03:00:00Z,2015-01-01T04:00:00+0100,5249.0,6454.0,38.0,NaN,64.0
...,...,...,...,...,...,...,...
50396,2020-09-30T19:00:00Z,2020-09-30T21:00:00+0200,6661.0,6656.0,NaN,NaN,1847.0
50397,2020-09-30T20:00:00Z,2020-09-30T22:00:00+0200,6336.0,6310.0,NaN,NaN,1723.0
50398,2020-09-30T21:00:00Z,2020-09-30T23:00:00+0200,5932.0,5813.0,NaN,NaN,1771.0
50399,2020-09-30T22:00:00Z,2020-10-01T00:00:00+0200,5628.0,5424.0,NaN,NaN,1779.0


In [65]:
df_AT__.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50401 entries, 0 to 50400
Data columns (total 7 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   utc_timestamp                         50401 non-null  object 
 1   cet_cest_timestamp                    50401 non-null  object 
 2   AT_load_actual_entsoe_transparency    50400 non-null  float64
 3   AT_load_forecast_entsoe_transparency  50400 non-null  float64
 4   AT_price_day_ahead                    32845 non-null  float64
 5   AT_solar_generation_actual            50339 non-null  float64
 6   AT_wind_onshore_generation_actual     50352 non-null  float64
dtypes: float64(5), object(2)
memory usage: 2.7+ MB
